# Week 9, Lecture 1: Lists, Loops and Dictionaries

Run each cell in order with **Shift + Enter**. Change the values and run again to see what happens.

The temperatures, rain, followers and posts below are made up.

## 1. `for` loops

A `for` loop takes each item in turn. It works on a string (one letter at a time) and on a list.

In [ ]:
for letter in "rain":
    print(letter)

r
a
i
n


In [ ]:
week_temps = [28, 31, 34, 29, 30, 26, 32]   # one week, in °C

for t in week_temps:
    if t >= 30:
        print(t)

31
34
30
32


## 2. `range`, `continue` and `break`

`range(n)` counts 0, 1, ..., n − 1. `continue` skips to the next item. `break` stops the loop.

In [ ]:
list(range(7))

[0, 1, 2, 3, 4, 5, 6]

In [ ]:
list(range(2, 5))

[2, 3, 4]

In [ ]:
for day in range(len(week_temps)):
    print(day, week_temps[day])

0 28
1 31
2 34
3 29
4 30
5 26
6 32


In [ ]:
for t in week_temps:
    if t < 30:
        continue          # skip the cooler days
    print("hot:", t)

hot: 31
hot: 34
hot: 30
hot: 32


In [ ]:
for t in week_temps:
    if t >= 34:
        print("heat alert:", t)
        break             # stop at the first very hot day
    print(t)

28
31
heat alert: 34


## 3. The accumulator pattern

Start with a value (0 for a total or a count), then update it once per item.

In [ ]:
def total_rain(rain_mm: list[float]) -> float:
    """total rain over the days, in mm"""
    run_total = 0
    for mm in rain_mm:
        run_total = run_total + mm
    return run_total


def count_hot_days(temps: list[int]) -> int:
    """how many days were 30°C or more"""
    count = 0
    for t in temps:
        if t >= 30:
            count = count + 1
    return count


def test_rain_and_heat():
    assert total_rain([2, 0, 5, 1]) == 8
    assert total_rain([]) == 0
    assert count_hot_days(week_temps) == 4
    assert count_hot_days([]) == 0


test_rain_and_heat()
print(total_rain([2, 0, 5, 1]), count_hot_days(week_temps))

8 4


## 4. List comprehensions

A short way to build a new list from an old one, with an optional `if` to keep only some items.

In [ ]:
hot_days = [t for t in week_temps if t >= 30]
hot_days

[31, 34, 30, 32]

In [ ]:
warmer = [t + 2 for t in week_temps]
warmer

[30, 33, 36, 31, 32, 28, 34]

## 5. Dictionaries

A dictionary maps **keys** to **values**. Here each account maps to its number of followers.

In [ ]:
followers = {"@climate_now": 15200, "@cityfarm": 830}
followers["@cityfarm"] = 845       # change a value
followers["@dr_rain"] = 4100       # add a new key and value
del followers["@cityfarm"]         # remove a key and its value
followers

{'@climate_now': 15200, '@dr_rain': 4100}

In [ ]:
followers["@climate_now"]

15200

In [ ]:
print(followers.keys())
print(followers.values())
print(followers.items())

dict_keys(['@climate_now', '@dr_rain'])
dict_values([15200, 4100])
dict_items([('@climate_now', 15200), ('@dr_rain', 4100)])


Asking for a key that is not there gives an error. `.get` gives a default instead, and `in` checks first.

In [ ]:
followers["@nobody"]          # KeyError, on purpose

KeyError: '@nobody'

In [ ]:
print(followers.get("@nobody", 0))
print("@dr_rain" in followers)
print(len(followers))

0
True
2


In [ ]:
for name in followers:          # a loop over a dictionary gives its keys
    print(name)

@climate_now
@dr_rain


## 6. A dictionary of posts

Each value can itself be a dictionary. Three common loops: **search**, **count** and **collect**.

In [ ]:
posts = {
    "p1": {"user": "@climate_now", "likes": 1250, "tags": ["climate"]},
    "p2": {"user": "@cityfarm", "likes": 85, "tags": ["food", "climate"]},
    "p3": {"user": "@dr_rain", "likes": 430, "tags": ["weather"]},
    "p4": {"user": "@dr_heat", "likes": 610, "tags": ["heat", "climate"]},
}
print(posts["p3"]["user"])
print(posts["p4"]["tags"])

@dr_rain
['heat', 'climate']


In [ ]:
# Search: which posts are tagged climate?
for post_id, details in posts.items():
    if "climate" in details["tags"]:
        print(post_id)

p1
p2
p4


In [ ]:
# Count: how many posts come from a "@dr_" account?
count = 0
for details in posts.values():
    if details["user"].startswith("@dr_"):
        count += 1
count

2

In [ ]:
# Collect: every tag, once each
all_tags = []
for details in posts.values():
    for tag in details["tags"]:
        if tag not in all_tags:
            all_tags.append(tag)
all_tags

['climate', 'food', 'weather', 'heat']

## 7. Dictionary or dataclass?

A dictionary takes any new key. A dataclass has a fixed set of fields, so a typo or a missing field is caught.

In [ ]:
from dataclasses import dataclass


@dataclass
class Post:
    user: str
    likes: int
    tags: list[str]


p3_as_dict = {"user": "@dr_rain", "likes": 430, "tags": ["weather"]}
p3_as_post = Post("@dr_rain", 430, ["weather"])

p3_as_dict["shares"] = 12          # fine: a new key
print(p3_as_dict["likes"], p3_as_post.likes)
p3_as_dict

430 430


{'user': '@dr_rain', 'likes': 430, 'tags': ['weather'], 'shares': 12}

In [ ]:
p3_as_post.shares                # AttributeError, on purpose

AttributeError: 'Post' object has no attribute 'shares'

A loop over a list of dataclass values:

In [ ]:
feed = [Post("@climate_now", 1250, ["climate"]),
        Post("@cityfarm", 85, ["food", "climate"]),
        Post("@dr_heat", 610, ["heat", "climate"])]


def popular_users(feed: list[Post], min_likes: int) -> list[str]:
    """the users of the posts with at least min_likes likes"""
    result = []
    for p in feed:
        if p.likes >= min_likes:
            result.append(p.user)
    return result


def test_popular_users():
    assert popular_users(feed, 500) == ["@climate_now", "@dr_heat"]
    assert popular_users(feed, 5000) == []
    assert popular_users([], 500) == []


test_popular_users()
popular_users(feed, 500)

['@climate_now', '@dr_heat']

## 8. Changing a list by its positions

`for t in temps` only reads the items. To change them, loop over the positions.

In [ ]:
temps = [28, 31, 34]
for i in range(len(temps)):
    temps[i] = temps[i] + 2
temps

[30, 33, 36]

---
# Class exercises

Write each function, then remove the `#` in front of the test call and run the cell. No output means every test passed.

### List iteration

**1.** `product_list(nums)` returns the product of all numbers in the list (1 for an empty list).

In [ ]:
def product_list(nums: list) -> float:
    """the product of all numbers in the list (1 for an empty list)"""
    pass   # your code here


def test_product_list():
    assert product_list([2, 3, 4]) == 24
    assert product_list([5]) == 5
    assert product_list([]) == 1

# test_product_list()

**2.** `count_occurrences(items, target)` returns how many times `target` appears in the list.

In [ ]:
def count_occurrences(items: list, target) -> int:
    """how many times target appears in items"""
    pass   # your code here


def test_count_occurrences():
    weather = ["sun", "rain", "rain", "cloud", "rain"]
    assert count_occurrences(weather, "rain") == 3
    assert count_occurrences([], "rain") == 0

# test_count_occurrences()

**3.** `filter_by_prefix(words, prefix)` returns the words that start with `prefix`. Hint: `s.startswith(prefix)`.

In [ ]:
def filter_by_prefix(words: list, prefix: str) -> list:
    """the words that start with prefix, in their original order"""
    pass   # your code here


def test_filter_by_prefix():
    tags = ["#climate", "#cop30", "#climatestrike", "#rain"]
    assert filter_by_prefix(tags, "#climate") == ["#climate", "#climatestrike"]
    assert filter_by_prefix(tags, "#snow") == []

# test_filter_by_prefix()

### Dictionary iteration

**4.** Make a dictionary that maps student IDs to names. Add a new student, update a name, and remove a student.

In [ ]:
students = {}   # your code here

**5.** Using `sessions` below, print the ID of every session that covers `"AI"`.

In [ ]:
sessions = {
    "CS101": {"title": "Introduction to AI", "speaker": "Dr. Martinez",
              "topics": ["AI", "Machine Learning"]},
    "CS102": {"title": "Deep Learning Techniques", "speaker": "Prof. Nguyen",
              "topics": ["Neural Networks", "Deep Learning"]},
    "CS103": {"title": "Quantum Computing Basics", "speaker": "Dr. Patel",
              "topics": ["Quantum", "Computing"]},
    "CS104": {"title": "Cybersecurity Trends", "speaker": "Ms. Lee",
              "topics": ["Security", "Networking"]},
}

# your code here

**6.** `unique_topics(sessions)` returns a list of every topic covered, once each.

In [ ]:
def unique_topics(sessions: dict) -> list:
    """every topic covered by the sessions, once each, in first-seen order"""
    pass   # your code here


def test_unique_topics():
    assert unique_topics({}) == []
    two = {"A1": {"topics": ["AI", "Ethics"]}, "A2": {"topics": ["Ethics", "Law"]}}
    assert unique_topics(two) == ["AI", "Ethics", "Law"]

# test_unique_topics()